# NB_patient_360

Builds `adb_medilake.gold.patient_360` per the STTM *Silver to Gold (CareSync Health Network)* v1.0, sheet `patient_360`.

* **Grain:** one row per patient
* **Driving table:** `adb_medilake.silver.patients` (all patients, no filters)
* **Joins:** LEFT JOIN `silver.hospitals` (registered_hospital_id = hospital_id), LEFT JOIN `silver.insurance_providers` (insurance_provider_id), LEFT JOIN lab results aggregated by patient_id
* **Load:** FULL rebuild with CREATE OR REPLACE TABLE ... AS SELECT

Implementation notes:
* `silver.lab_results` contains exact duplicate `lab_result_id` rows (re-loaded records), so lab rows are de-duplicated on `lab_result_id` (latest `update_timestamp`) to respect the documented grain of one row per lab test.
* `silver.patients.date_of_birth` is a timestamp, so it is cast to DATE.

In [0]:
%sql
-- Full rebuild of gold.patient_360 (one row per patient)
CREATE OR REPLACE TABLE adb_medilake.gold.patient_360
COMMENT 'Patient 360: one row per patient with hospital, insurance and lab test summary (STTM silver to gold v1.0)'
AS
WITH lab AS (
  -- De-duplicate lab results to one row per lab_result_id
  SELECT *
  FROM adb_medilake.silver.lab_results
  QUALIFY row_number() OVER (PARTITION BY lab_result_id ORDER BY update_timestamp DESC) = 1
),
lab_agg AS (
  SELECT
    patient_id,
    COUNT(*)                              AS total_tests,
    COUNT_IF(result_status = 'Abnormal')  AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')  AS critical_test_count,
    MAX(test_date)                        AS last_test_date
  FROM lab
  GROUP BY patient_id
)
SELECT
  p.patient_id,
  p.first_name,
  p.last_name,
  CAST(p.date_of_birth AS DATE)                                                AS date_of_birth,
  CAST(FLOOR(datediff(current_date(), CAST(p.date_of_birth AS DATE)) / 365.25) AS INT) AS age,
  p.gender,
  p.city,
  p.state,
  p.registered_hospital_id                                                     AS hospital_id,
  h.hospital_name,
  p.insurance_provider_id,
  i.provider_name                                                              AS insurance_provider_name,
  CAST(COALESCE(l.total_tests, 0) AS INT)                                      AS total_tests,
  CAST(COALESCE(l.abnormal_test_count, 0) AS INT)                              AS abnormal_test_count,
  CAST(COALESCE(l.critical_test_count, 0) AS INT)                              AS critical_test_count,
  l.last_test_date
FROM adb_medilake.silver.patients p
LEFT JOIN adb_medilake.silver.hospitals h
  ON p.registered_hospital_id = h.hospital_id
LEFT JOIN adb_medilake.silver.insurance_providers i
  ON p.insurance_provider_id = i.insurance_provider_id
LEFT JOIN lab_agg l
  ON p.patient_id = l.patient_id;

In [0]:
# Return the record count of the rebuilt gold table as the notebook exit value
updated_df = spark.table("adb_medilake.gold.patient_360")
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))